# `microstructure_demo` — Iterative Improvement Loop

Iteratively raises the CV R² on Cycle 1 HoldingTemp / HoldingTime by stacking the strategies discussed in the analysis (log-transformed time, per-target single-output models, stratified KFolds). Caches are read **read-only** from the demo notebook's `data/image_cache_*.npz` and `features/morph_features_c1.npz` — nothing is rebuilt.

All real work lives in [`scripts/iterate_demo.py`](../scripts/iterate_demo.py). Each iteration writes:

- `runs/iterate_demo/<run_id>/iteration_<n>.md` — what was tried, code diff, deltas, winner
- `runs/iterate_demo/<run_id>/summary.md` — running cumulative table
- `runs/iterate_demo/<run_id>/history.csv` — long-format row per (iteration, strategy, target)

Ordering of Tier-1 strategies (tried left-to-right per iteration; the largest positive Δ-mean-R² wins and is folded into the baseline):

1. `log_time` — wrap HoldingTime with `TransformedTargetRegressor(log1p)` so linear MSE doesn't over-weight long-time samples.
2. `per_target` — train two single-output GBRs instead of `MultiOutputRegressor` wrapping a joint GBR.
3. `stratify_alloy` — `StratifiedKFold` by alloy so every alloy is represented in train and test.
4. `stratify_temp_bin` — `StratifiedKFold` by HoldingTemp quantile-bin so rare setpoints aren't isolated.

Stop criterion: no remaining strategy clears `--min-delta` (default `0.005`) on the mean of (HoldingTemp R², HoldingTime R²).

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path(os.path.abspath('..'))
script = REPO / 'scripts' / 'iterate_demo.py'
print(f'Running: {script}')

result = subprocess.run(
    [sys.executable, str(script),
     '--csv',       str(REPO / 'datasets' / 'metadata_latest.csv'),
     '--backbone',  'dinov2_vitb14',
     '--cache-dir', str(REPO / 'data'),
     '--morph-cache', str(REPO / 'features' / 'morph_features_c1.npz'),
     '--tier',      '1',
     '--min-delta', '0.005',
     '--cv-repeats', '3',
     '--final-cv-repeats', '10',
     # Carve a 15% held-out test split before iteration; the winning
     # configuration is then refit on the remaining 85% and evaluated
     # on the held-out rows. Produces an honest test-set R²/MAE/RMSE
     # plus a predicted-vs-actual scatter for both targets.
     '--with-test-split', '0.15'],
    cwd=str(REPO),
)
print(f'\nExit code: {result.returncode}')

In [ ]:
# Show the latest run's summary inline.
from pathlib import Path
import os
REPO = Path(os.path.abspath('..'))
run_root = REPO / 'runs' / 'iterate_demo'
latest = None
if run_root.exists():
    runs = sorted([p for p in run_root.iterdir() if p.is_dir()])
    if runs:
        latest = runs[-1]
        print(f'Latest run: {latest}')
        summary = latest / 'summary.md'
        if summary.exists():
            print()
            print(summary.read_text())

In [ ]:
# Show the per-iteration markdown logs.
if latest is not None:
    for it_md in sorted(latest.glob('iteration_*.md')):
        print(f"\n{'=' * 70}")
        print(f'  {it_md.name}')
        print(f"{'=' * 70}\n")
        print(it_md.read_text())

In [ ]:
# ── Held-out test set — predicted vs actual + headline metrics ───────────────
# Renders the test-set scatter the script wrote and prints the headline numbers
# from the manifest. These reflect generalisation to data the iteration loop
# never saw — the most defensible single-number summary in this run.
if latest is not None:
    import json as _json
    from IPython.display import Image, display, Markdown

    _png = latest / 'test_predicted_vs_actual.png'
    if _png.exists():
        display(Image(filename=str(_png)))
    else:
        print(f'No test-set scatter at {_png}. Re-run with --with-test-split.')

    _manifest = latest / 'manifest.json'
    if _manifest.exists():
        m = _json.loads(_manifest.read_text())
        if m.get('test_split_frac') is not None:
            display(Markdown(
                f"""
**Held-out test set summary** (n = {m['test_n']}, frac = {m['test_split_frac']})

| Target       | R²                          | MAE                              | RMSE                              |
|--------------|-----------------------------|----------------------------------|-----------------------------------|
| HoldingTemp  | {m['test_temp_r2']:+.4f}    | {m['test_temp_mae']:.2f} °C      | {m['test_temp_rmse']:.2f} °C      |
| HoldingTime  | {m['test_time_r2']:+.4f}    | {m['test_time_mae']:.2f} min     | {m['test_time_rmse']:.2f} min     |
| **Mean**     | **{m['test_mean_r2']:+.4f}** |                                  |                                   |

Winning configuration: `{m['stack']}`
"""
            ))
